In [1]:
import re
import sys
import json
from pathlib import Path
import pdfplumber

ROW_PATTERN = re.compile(
    r'^(?P<itm>\d+)\s+'
    r'(?P<dos>\d{2}/\d{2}/\d{2})\s+'
    r'(?P<code>D\d{4})(?:\s+\d{1,2})?(?:\s+[A-Za-z]{1,3})?\s+'
    r'(?P<qty1>\d+)\s+'
    r'(?P<billed>\$[\d,.]+)\s+'
    r'(?P<qty2>\d+)\s+'
    r'(?P<allowed>\$[\d,.]+)\s+'
    r'(?P<disallow>\$[\d,.]+)\s+'
    r'(?P<payable>\$[\d,.]+)\s+'
    r'(?P<copay>\$[\d,.]+)\s+'
    r'(?P<coins>\$[\d,.]+)\s+'
    r'(?P<deduct>\$[\d,.]+)\s+'
    r'(?P<patientpay>\$[\d,.]+)\s+'
    r'(?P<otherinsur>\$[\d,.]+)\s+'
    r'(?P<net>\$[\d,.]+)\s+'
    r'(?P<exc>.+)$',
    re.MULTILINE
)

TOTALS_PATTERN = re.compile(
    r'^\$(?P<billed>[\d,.]+)\s+\$(?P<allowed>[\d,.]+)\s+\$(?P<disallow>[\d,.]+)\s+'
    r'\$(?P<payable>[\d,.]+)\s+\$(?P<copay>[\d,.]+)\s+\$(?P<coins>[\d,.]+)\s+'
    r'\$(?P<deduct>[\d,.]+)\s+\$(?P<patientpay>[\d,.]+)\s+\$(?P<otherinsur>[\d,.]+)\s+'
    r'\$(?P<net>[\d,.]+)$',
    re.MULTILINE
)

ITEM_NOTE_PATTERN = re.compile(
    r'ITEM:\s*(?P<itm>\d+)\s+'
    r'(?:Service Payment Notes:\s*(?P<notes>.*?)'
    r'(?=\n(?:ITEM:|Authorizations:|Patient Name:|Ref #:|Payee ID:|BILLED ALLOWED|~)|\Z)'
    r'|Exception Code:\s*(?P<exc_code>\S+)\s+(?P<reason>.*?)'
    r'(?=\n(?:ITEM:|Authorizations:|Patient Name:|Ref #:|Payee ID:|BILLED ALLOWED|~)|\Z))',
    re.DOTALL
)

def _is_patient_page(text):
    return "Patient Name:" in text
def extract_remittance_pdf(pdf_path):
    pdf_path = Path(pdf_path)

    with pdfplumber.open(pdf_path) as pdf:
        pages = pdf.pages
        kept_texts = []

        for page in pages[2:]:
            text = page.extract_text() or ""
            if _is_patient_page(text):
                kept_texts.append(text)

    full_text = "\n".join(kept_texts)

    provider_m = re.search(r'Provider Name:\s*(.+?)\s+Encounter', full_text)
    provider_name = provider_m.group(1).strip() if provider_m else None

    raw_blocks = re.split(r'(?=Patient Name:)', full_text)
    blocks = [b for b in raw_blocks if b.strip().startswith("Patient Name:")]

    patients = [_extract_patient_block(b) for b in blocks]

    return {
        "file_name": pdf_path.name,
        "provider_name": provider_name,
        "confidence_score": "100",
        "patients": patients
    }

def _extract_patient_block(block):
    name_m = re.search(r'Patient Name:\s*(.+?)\s+Provider Name:', block)
    dob_m = re.search(r'DOB:\s*(\d{2}/\d{2}/\d{4})', block)
    prov_m = re.search(r'Provider Name:\s*(.+?)\s+Encounter', block)

    item_notes = {}

    for m in ITEM_NOTE_PATTERN.finditer(block):
        itm = m.group("itm")

        if m.group("exc_code"):
            reason_clean = re.sub(r'\s+', ' ', m.group("reason")).strip()
            item_notes[itm] = {
                "denial_status": "Denied",
                "denial_reason": f'{m.group("exc_code")} - {reason_clean}'.strip(" -")
            }
        else:
            item_notes[itm] = {
                "denial_status": "Not Denied",
                "denial_reason": None
            }

    services = []

    for m in ROW_PATTERN.finditer(block):
        itm = m.group("itm")

        note = item_notes.get(
            itm,
            {
                "denial_status": "Not Denied",
                "denial_reason": None
            }
        )

        services.append({
            "item": itm,
            "dos": m.group("dos"),
            "code": m.group("code"),
            "billed_amount": m.group("billed"),
            "allowed_amount": m.group("allowed"),
            "disallow_amount": m.group("disallow"),
            "payable_amount": m.group("payable"),
            "copay_amount": m.group("copay"),
            "coins_amount": m.group("coins"),
            "deduct_amount": m.group("deduct"),
            "patient_pay": m.group("patientpay"),
            "other_insur": m.group("otherinsur"),
            "net_amount": m.group("net"),
            "exc_code": m.group("exc").strip(),
            "denial_status": note["denial_status"],
            "denial_reason": note["denial_reason"]
        })

    totals_m = TOTALS_PATTERN.search(block)

    if totals_m:
        totals = {
            "total_billed_amount": f'${totals_m.group("billed")}',
            "total_allowed_amount": f'${totals_m.group("allowed")}',
            "total_disallow_amount": f'${totals_m.group("disallow")}',
            "total_payable_amount": f'${totals_m.group("payable")}',
            "total_copay_amount": f'${totals_m.group("copay")}',
            "total_coins_amount": f'${totals_m.group("coins")}',
            "total_deduct_amount": f'${totals_m.group("deduct")}',
            "total_patient_pay": f'${totals_m.group("patientpay")}',
            "total_other_insur": f'${totals_m.group("otherinsur")}',
            "total_net_amount": f'${totals_m.group("net")}'
        }
    else:
        totals = {
            k: None for k in [
                "total_billed_amount",
                "total_allowed_amount",
                "total_disallow_amount",
                "total_payable_amount",
                "total_copay_amount",
                "total_coins_amount",
                "total_deduct_amount",
                "total_patient_pay",
                "total_other_insur",
                "total_net_amount"
            ]
        }

    return {
        "patient_name": name_m.group(1).strip() if name_m else None,
        # "provider_name": prov_m.group(1).strip() if prov_m else None,
        "dob": dob_m.group(1) if dob_m else None,
        "services": services,
        "totals": totals
    }
def flatten_for_table(pdf_result):
    rows = []

    for patient in pdf_result["patients"]:
        base = {
            "file_name": pdf_result["file_name"],
            "patient_name": patient["patient_name"],
            "provider_name": pdf_result["provider_name"],
            "dob": patient["dob"],
            **patient["totals"]
        }

        if patient["services"]:
            for svc in patient["services"]:
                row = dict(base)
                row.update(svc)
                rows.append(row)
        else:
            rows.append(base)

    return rows

def run_batch(input_dir, output_path):
    input_dir = Path(input_dir)
    output_path = Path(output_path)

    pdf_files = sorted(input_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return

    all_rows = []
    all_results = []
    errors = []

    for pdf_path in pdf_files:
        try:
            result = extract_remittance_pdf(pdf_path)
            all_results.append(result)
            all_rows.extend(flatten_for_table(result))

            n_patients = len(result["patients"])
            n_services = sum(
                len(p["services"]) for p in result["patients"]
            )
            n_denied = sum(
                1
                for p in result["patients"]
                for s in p["services"]
                if s["denial_status"] == "Denied"
            )

            flag = "" if n_patients else "  <-- NO PATIENT BLOCKS FOUND, CHECK THIS FILE"

            print(
                f"OK   {pdf_path.name}: "
                f"{n_patients} patient(s), "
                f"{n_services} service(s), "
                f"{n_denied} denied{flag}"
            )

        except Exception as e:
            errors.append((pdf_path.name, str(e)))
            print(f"FAIL {pdf_path.name}: {e}")

    columns = [
        "file_name",
        "patient_name",
        "provider_name",
        "dob",
        "item",
        "dos",
        "code",
        "billed_amount",
        "allowed_amount",
        "disallow_amount",
        "payable_amount",
        "copay_amount",
        "coins_amount",
        "deduct_amount",
        "patient_pay",
        "other_insur",
        "net_amount",
        "exc_code",
        "denial_status",
        "denial_reason",
        "total_billed_amount",
        "total_allowed_amount",
        "total_disallow_amount",
        "total_payable_amount",
        "total_copay_amount",
        "total_coins_amount",
        "total_deduct_amount",
        "total_patient_pay",
        "total_other_insur",
        "total_net_amount"
    ]

    if output_path.suffix.lower() == ".xlsx":
        import openpyxl
        from openpyxl.utils import get_column_letter

        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "Remittance"

        ws.append(columns)

        for row in all_rows:
            ws.append([row.get(c, "") for c in columns])

        for i, col in enumerate(columns, start=1):
            max_len = max(
                [len(col)] +
                [len(str(r.get(col, ""))) for r in all_rows]
            )
            ws.column_dimensions[
                get_column_letter(i)
            ].width = min(max_len + 2, 40)

        wb.save(output_path)

    else:
        import csv

        with open(
            output_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=columns
            )
            writer.writeheader()

            for row in all_rows:
                writer.writerow(row)

    json_path = output_path.with_suffix(".json")

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            all_results,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"\nDone. {len(pdf_files)} PDF(s) processed, {len(errors)} failed.")
    print(f"Table written to:  {output_path}")
    print(f"Raw JSON written to: {json_path}")

    if errors:
        print("\nFiles that failed:")

        for name, err in errors:
            print(f"  - {name}: {err}")

def run_single(pdf_path, output_root="./Test_1"):
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}"
        )

    result = extract_remittance_pdf(pdf_path)

    folder_name = pdf_path.stem
    output_folder = Path(output_root) / folder_name
    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    json_path = output_folder / f"{folder_name}.json"

    with open(
        json_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            [result],
            f,
            indent=2,
            ensure_ascii=False
        )

    n_patients = len(result["patients"])

    n_services = sum(
        len(p["services"])
        for p in result["patients"]
    )

    n_denied = sum(
        1
        for p in result["patients"]
        for s in p["services"]
        if s["denial_status"] == "Denied"
    )

    print("=" * 70)
    print(f"PDF      : {pdf_path.name}")
    print(f"Patients : {n_patients}")
    print(f"Services : {n_services}")
    print(f"Denied   : {n_denied}")
    print(f"JSON     : {json_path}")
    print("=" * 70)

    print("\nDENIED SERVICES")
    print("-" * 70)

    for patient in result["patients"]:
        for service in patient["services"]:
            if service["denial_status"] == "Denied":
                print(f"Patient : {patient['patient_name']}")
                print(f"Item    : {service['item']}")
                print(f"Code    : {service['code']}")
                print(f"EXC     : {service['exc_code']}")
                print(f"Reason  : {service['denial_reason']}")
                print("-" * 70)

    return result, json_path

In [4]:
pdf_path = r"/home/cipl/users/Yashwanth/Qodoro_OCR/phase_1/EOBs/United_health/UHC_goverment_pdfs/487844687.pdf"
result, json_path = run_single(pdf_path)

PDF      : 487844687.pdf
Patients : 4
Services : 15
Denied   : 4
JSON     : Test_1/487844687/487844687.json

DENIED SERVICES
----------------------------------------------------------------------
Patient : HORISBERGER, EMMA
Item    : 2
Code    : D2394
EXC     : 1032
Reason  : 1032 - This service is not valid for the tooth number submitted
----------------------------------------------------------------------
Patient : HORISBERGER, EMMA
Item    : 3
Code    : D2391
EXC     : 1032
Reason  : 1032 - This service is not valid for the tooth number submitted
----------------------------------------------------------------------
Patient : HORISBERGER, EMMA
Item    : 4
Code    : D2392
EXC     : 1032
Reason  : 1032 - This service is not valid for the tooth number submitted
----------------------------------------------------------------------
Patient : HORISBERGER, EMMA
Item    : 5
Code    : D2392
EXC     : 1032
Reason  : 1032 - This service is not valid for the tooth number submitted
-----------

In [9]:
pdf_path = r"/home/cipl/users/Jeeva/Phase_2_pdf/UPMC/$1,431.77.pdf"
result, json_path = run_single(pdf_path)

PDF      : $1,431.77.pdf
Patients : 11
Services : 56
Denied   : 4
JSON     : Test_1/$1,431.77/$1,431.77.json

DENIED SERVICES
----------------------------------------------------------------------
Patient : CASTRO, SARAH
Item    : 5
Code    : D1310
EXC     : PR204 N130
Reason  : 1039 - Service Not a Covered Benefit. (PR204 N130)
----------------------------------------------------------------------
Patient : CASTRO, SARAH
Item    : 6
Code    : D1330
EXC     : PR204 N130
Reason  : 1039 - Service Not a Covered Benefit. (PR204 N130)
----------------------------------------------------------------------
Patient : VASQUEZ ALMONT, DIEGO
Item    : 5
Code    : D1310
EXC     : PR204 N130
Reason  : 1039 - Service Not a Covered Benefit. (PR204 N130)
----------------------------------------------------------------------
Patient : VASQUEZ ALMONT, DIEGO
Item    : 6
Code    : D1330
EXC     : PR204 N130
Reason  : 1039 - Service Not a Covered Benefit. (PR204 N130)
--------------------------------------